# Building a MiniGPT Language Model from Scratch for Next Token Prediction

This Colab notebook trains a small Transformer language model on **Tiny Shakespeare** for next-token prediction and live text generation.

## 1. Dataset and Training Setup

We use **Tiny Shakespeare** (about 1 MB):
- Small dataset
- Repetitive language patterns
- Learns quickly on Colab GPU

Recommended setup used here:

| Parameter | Value |
|---|---|
| Model size | ~20M parameters |
| Block size | 128 |
| Batch size | 32 |
| Epochs | 8 (inside recommended 5-10) |
| Learning rate | 3e-4 |

Expected training time on Colab T4 GPU: about **20-40 minutes**.

## 2. Demo Narrative (For Presentation)

### Step 1 - Explain the concept

**Say this:**

"This model learns next token prediction. Given a sequence of tokens, it predicts the most probable next token."

Example:
- Input: `The king was`
- Prediction idea: `The king was angry`

### Step 2 - Show generation
Prompt example: `ROMEO:`

### Step 3 - Explain architecture

Simple flow:

Text -> Tokenizer -> Token Embeddings -> Transformer Blocks -> Linear Layer -> Next Token Prediction

Talk briefly about:
- Self-attention
- Transformer blocks
- Training on token sequences

### If asked about limitations

"Due to limited GPU compute and dataset size, the model cannot achieve large-scale GPT performance, but the architecture and training pipeline replicate the core mechanism used in modern language models."

In [ ]:
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm.auto import tqdm

seed = 1337
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download Tiny Shakespeare
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt

text_path = Path("input.txt")
text = text_path.read_text(encoding="utf-8")
print(f"Dataset path: {text_path.resolve()}")
print(f"Characters in dataset: {len(text):,}")
print("Preview:")
print(text[:500])

In [ ]:
# Character-level tokenizer (token = character)
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
val_data = data[split_idx:]

print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")

In [ ]:
# Recommended hyperparameters
block_size = 128
batch_size = 32
learning_rate = 3e-4
epochs = 8
eval_interval = 200
eval_batches = 40
dropout = 0.1

# ~20M parameter model
n_embd = 416
n_head = 8
n_layer = 10

steps_per_epoch = max(1, len(train_data) // (batch_size * block_size))
max_iters = steps_per_epoch * epochs

def get_batch(split: str):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data_source) - block_size - 1, (batch_size,))
    x = torch.stack([data_source[i : i + block_size] for i in ix])
    y = torch.stack([data_source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print(f"steps_per_epoch: {steps_per_epoch}")
print(f"total training steps: {max_iters}")

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_drop(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, n_embd)
        self.pos_embed = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([
            TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self._encode = None
        self._decode = None

    def set_tokenizer(self, encode_fn, decode_fn):
        self._encode = encode_fn
        self._decode = decode_fn

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}.")

        pos = torch.arange(0, T, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=100, temperature=1.0, top_k=None):
        if isinstance(prompt, str):
            if self._encode is None or self._decode is None:
                raise ValueError("Tokenizer functions are not set. Call model.set_tokenizer first.")
            idx = torch.tensor([self._encode(prompt)], dtype=torch.long, device=next(self.parameters()).device)
            return_text = True
        elif torch.is_tensor(prompt):
            idx = prompt.to(next(self.parameters()).device)
            return_text = False
        else:
            raise TypeError("prompt must be a string or a tensor of token ids")

        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None and top_k > 0:
                k = min(top_k, logits.size(-1))
                v, _ = torch.topk(logits, k)
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        if return_text:
            return self._decode(idx[0].tolist())
        return idx

In [ ]:
model = MiniGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    dropout=dropout,
).to(device)
model.set_tokenizer(encode, decode)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params / 1e6:.2f}M")

In [ ]:
@torch.no_grad()
def estimate_loss(eval_batches=40):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_batches)
        for k in range(eval_batches):
            xb, yb = get_batch(split)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

train_losses = []
val_losses = []
eval_steps = []

start = time.time()
model.train()

for step in tqdm(range(max_iters), desc="Training"):
    xb, yb = get_batch("train")

    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    if step % eval_interval == 0 or step == max_iters - 1:
        stats = estimate_loss(eval_batches=eval_batches)
        eval_steps.append(step)
        train_losses.append(stats["train"])
        val_losses.append(stats["val"])
        print(
            f"step {step:4d}/{max_iters} | train loss {stats['train']:.4f} | val loss {stats['val']:.4f}"
        )

elapsed = time.time() - start
print(f"Training completed in {elapsed / 60:.1f} minutes")

In [ ]:
# Bonus: show loss decreasing
plt.figure(figsize=(10, 4))
plt.plot(eval_steps, train_losses, label="Train loss")
plt.plot(eval_steps, val_losses, label="Val loss")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("MiniGPT on Tiny Shakespeare")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Simple generation code for demo
prompt = "ROMEO:"
generated = model.generate(prompt, max_new_tokens=100)
print(generated)

In [ ]:
# If output looks weak, increase randomness and top-k filtering
prompt = "ROMEO:"
generated = model.generate(prompt, max_new_tokens=160, temperature=0.8, top_k=40)
print(generated)

In [ ]:
# Optional: save model checkpoint from Colab
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_layer": n_layer,
        "n_head": n_head,
        "n_embd": n_embd,
        "stoi": stoi,
        "itos": itos,
    },
    "mini_gpt_tinyshakespeare.pt",
)
print("Saved checkpoint: mini_gpt_tinyshakespeare.pt")